# Figure 1: Prehospital recognition gap
This notebook reproduces Figure 1A (false‑negative case timeline) and Figure 1B (sensitivity across transport phases).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

# Set global style
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Helvetica', 'Arial']
rcParams['axes.linewidth'] = 1.5
rcParams['xtick.major.width'] = 1.5
rcParams['ytick.major.width'] = 1.5

# Color scheme
color_trauma = '#E64B35'      # red (Trauma-Former / TIC)
color_lstm = '#4DBBD5'        # blue (LSTM / control)
color_shock = '#00A087'       # green (shock index)
color_hr = '#D55E00'          # orange (heart rate)
color_bp = '#0072B2'          # dark blue (blood pressure / pulse pressure)

# Data directory – adjust if needed
data_dir = './data'

In [ ]:
# Load data
df_case = pd.read_csv(f'{data_dir}/Fig1_FalseNegative_Case.csv')
df_sens = pd.read_csv(f'{data_dir}/Fig1_Time_Sensitivity.csv')

# Compute pulse pressure
df_case['PP'] = df_case['SBP'] - df_case['DBP']

# Find first time where risk probability >= 0.8
threshold_risk = 0.8
idx_first_above = df_case[df_case['Risk_Probability'] >= threshold_risk].index.min()
if pd.notna(idx_first_above):
    first_time = df_case.loc[idx_first_above, 'Time_min']
else:
    first_time = None

In [ ]:
# Create figure
fig = plt.figure(figsize=(12, 5))
gs = fig.add_gridspec(1, 2, width_ratios=[1.6, 1], wspace=0.3)

# Panel A: false-negative case timeline
ax1 = fig.add_subplot(gs[0])
ax1.set_xlabel('Time (min)', fontsize=11)
ax1.set_ylabel('HR (bpm) / PP (mmHg)', fontsize=11, color='black')

l1 = ax1.plot(df_case['Time_min'], df_case['Heart_Rate'], color=color_hr, linewidth=2, label='HR')
l2 = ax1.plot(df_case['Time_min'], df_case['PP'], color=color_bp, linewidth=2, label='PP')
ax1.tick_params(axis='y', labelcolor='black')

ax2 = ax1.twinx()
ax2.set_ylabel('Shock Index / Risk Probability', fontsize=11, color='black')

l3 = ax2.plot(df_case['Time_min'], df_case['Shock_Index'], color=color_shock, linestyle='--', linewidth=2, label='Shock Index')
l4 = ax2.plot(df_case['Time_min'], df_case['Risk_Probability'], color=color_trauma, linewidth=2.5, label='Risk Probability')
ax2.tick_params(axis='y', labelcolor='black')

ax2.axhline(y=0.8, color=color_trauma, linestyle=':', linewidth=1.5, alpha=0.7)
ax2.axhline(y=0.9, color=color_shock, linestyle=':', linewidth=1.5, alpha=0.7)
if first_time is not None:
    ax1.axvline(x=first_time, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
    ax1.text(first_time+0.5, 110, f'First alert\n({first_time:.0f} min)', fontsize=9, color=color_trauma)

lines = l1 + l2 + l3 + l4
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=9, frameon=False)

ax1.set_xlim(0, 30)
ax1.set_ylim(30, 120)
ax2.set_ylim(0, 1.2)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.set_title('A  False-negative case: delayed shock index alert', fontsize=12, loc='left', weight='bold')

# Panel B: sensitivity across transport phases
ax3 = fig.add_subplot(gs[1])
time_points = np.arange(len(df_sens))
time_labels = df_sens['Time_Window']
ax3.plot(time_points, df_sens['Model_Sensitivity'], color=color_trauma, marker='o', linewidth=2, markersize=6, label='Trauma-Former')
ax3.plot(time_points, df_sens['ShockIndex_Sensitivity'], color=color_shock, marker='s', linewidth=2, markersize=6, label='Shock Index')
ax3.set_xticks(time_points)
ax3.set_xticklabels(time_labels, rotation=45, ha='right', fontsize=9)
ax3.set_xlabel('Time Window', fontsize=11)
ax3.set_ylabel('Sensitivity', fontsize=11)
ax3.set_ylim(0, 1.05)
ax3.grid(True, linestyle='--', alpha=0.5)
ax3.legend(loc='lower right', fontsize=9, frameon=False)
ax3.set_title('B  Sensitivity across transport phases', fontsize=12, loc='left', weight='bold')

plt.tight_layout()
plt.savefig('Figure1.tiff', dpi=300, format='tiff', bbox_inches='tight')
plt.show()